<a href="https://colab.research.google.com/github/nigamankit7/chronodex/blob/main/Chronodex_Planner_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⏱️ Chronodex Planner Generator
Run the cells below to install the required LaTeX engines and generate your custom, print-ready Chronodex daily planners.

In [16]:
# Install LaTeX environment (This takes about 1-2 minutes)
!sudo apt-get update
!sudo apt-get install -y texlive-latex-extra texlive-fonts-recommended texlive-pictures

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
texlive-fonts-recommended is already the newest version (2021.20220204-1).
texlive-latex-extra

In [43]:
# @title ⚙️ Chronodex Configuration & Generation
# @markdown Fill in your preferences and run this cell to generate the PDF.

START_DATE = "2026-06-05"  # @param {type:"string"}
END_DATE = "2026-06-30"    # @param {type:"string"}
FILE_NAME = "Chronodex_Planner_Final"  # @param {type:"string"}
THEME_SELECTION = "Corporate Slate"  # @param ["Minimalist", "Ocean", "Earth", "Corporate Slate", "Elegant Burgundy", "Matcha Green", "Midnight Blue"]

import datetime
import subprocess
import os

# --- Theme Engine ---
THEMES = {
    "Minimalist": {
        "chrono_color": "black!80",
        "month_color": "black!60",
        "style": r"""chronodex/mytheme/.style={
        24 hours, tick middle code={(0:\pgfkeysvalueof{/tikz/chronodex/tick middle start}) node[circle, inner sep=1.25pt, fill=black!60] {}},
        base ring/.append style={black!80, very thick}, base segments/.append style={black!4},
        primary ring/.append style={black!80, thick}, primary tick/.append style={black!80, thick}, secondary tick/.append style={black!50, thick},
        segment 6 to 7 am/.append style={black!20}, segment 7 to 8 am/.append style={black!15}, segment 8 to 9 am/.append style={black!10},
        segment 9 to 10 pm/.append style={black!20}, segment 10 to 11 pm/.append style={black!15}, segment 11 pm to 12 am/.append style={black!10},
        label/.append style={text=black!80, font=\sffamily}, day/.append style={text=black}, weekday/.append style={text=black!60}
    }"""
    },
    "Ocean": {
        "chrono_color": "blue!60!black",
        "month_color": "cyan!70!black",
        "style": r"""chronodex/mytheme/.style={
        24 hours, tick middle code={(0:\pgfkeysvalueof{/tikz/chronodex/tick middle start}) node[circle, inner sep=1.25pt, fill=cyan!70!black] {}},
        base ring/.append style={blue!60!black, very thick}, base segments/.append style={cyan!5},
        primary ring/.append style={blue!60!black, thick}, primary tick/.append style={blue!70!black, thick}, secondary tick/.append style={cyan!70!black, thick},
        segment 6 to 7 am/.append style={cyan!30}, segment 7 to 8 am/.append style={cyan!20}, segment 8 to 9 am/.append style={cyan!10},
        segment 9 to 10 pm/.append style={blue!30}, segment 10 to 11 pm/.append style={blue!20}, segment 11 pm to 12 am/.append style={blue!10},
        label/.append style={text=blue!60!black, font=\sffamily}, day/.append style={text=blue!70!black}, weekday/.append style={text=cyan!70!black}
    }"""
    },
    "Earth": {
        "chrono_color": "brown!70!black",
        "month_color": "olive!80!black",
        "style": r"""chronodex/mytheme/.style={
        24 hours, tick middle code={(0:\pgfkeysvalueof{/tikz/chronodex/tick middle start}) node[circle, inner sep=1.25pt, fill=olive!80!black] {}},
        base ring/.append style={brown!70!black, very thick}, base segments/.append style={orange!5},
        primary ring/.append style={brown!70!black, thick}, primary tick/.append style={brown!80!black, thick}, secondary tick/.append style={olive!80!black, thick},
        segment 6 to 7 am/.append style={orange!30}, segment 7 to 8 am/.append style={orange!20}, segment 8 to 9 am/.append style={orange!10},
        segment 9 to 10 pm/.append style={olive!30}, segment 10 to 11 pm/.append style={olive!20}, segment 11 pm to 12 am/.append style={olive!10},
        label/.append style={text=brown!70!black, font=\sffamily}, day/.append style={text=brown!80!black}, weekday/.append style={text=olive!80!black}
    }"""
    },
    "Corporate Slate": {
        "chrono_color": "darkgray",
        "month_color": "teal!60!black",
        "style": r"""chronodex/mytheme/.style={
        24 hours, tick middle code={(0:\pgfkeysvalueof{/tikz/chronodex/tick middle start}) node[circle, inner sep=1.25pt, fill=teal!60!black] {}},
        base ring/.append style={darkgray, very thick}, base segments/.append style={gray!5},
        primary ring/.append style={darkgray, thick}, primary tick/.append style={darkgray, thick}, secondary tick/.append style={gray!70, thick},
        segment 6 to 7 am/.append style={gray!30}, segment 7 to 8 am/.append style={gray!20}, segment 8 to 9 am/.append style={gray!10},
        segment 9 to 10 pm/.append style={teal!20}, segment 10 to 11 pm/.append style={teal!15}, segment 11 pm to 12 am/.append style={teal!5},
        label/.append style={text=darkgray, font=\sffamily}, day/.append style={text=black}, weekday/.append style={text=teal!60!black}
    }"""
    },
    "Elegant Burgundy": {
        "chrono_color": "purple!50!black",
        "month_color": "purple!70!black",
        "style": r"""chronodex/mytheme/.style={
        24 hours, tick middle code={(0:\pgfkeysvalueof{/tikz/chronodex/tick middle start}) node[circle, inner sep=1.25pt, fill=purple!50!black] {}},
        base ring/.append style={purple!40!black, very thick}, base segments/.append style={purple!3},
        primary ring/.append style={purple!50!black, thick}, primary tick/.append style={purple!50!black, thick}, secondary tick/.append style={purple!30!black, thick},
        segment 6 to 7 am/.append style={orange!20}, segment 7 to 8 am/.append style={orange!10}, segment 8 to 9 am/.append style={orange!5},
        segment 9 to 10 pm/.append style={purple!30}, segment 10 to 11 pm/.append style={purple!20}, segment 11 pm to 12 am/.append style={purple!10},
        label/.append style={text=purple!50!black, font=\sffamily}, day/.append style={text=purple!60!black}, weekday/.append style={text=purple!40!black}
    }"""
    },
    "Matcha Green": {
        "chrono_color": "green!40!black",
        "month_color": "olive!60!black",
        "style": r"""chronodex/mytheme/.style={
        24 hours, tick middle code={(0:\pgfkeysvalueof{/tikz/chronodex/tick middle start}) node[circle, inner sep=1.25pt, fill=olive!60!black] {}},
        base ring/.append style={green!30!black, very thick}, base segments/.append style={green!5},
        primary ring/.append style={green!40!black, thick}, primary tick/.append style={green!40!black, thick}, secondary tick/.append style={green!50, thick},
        segment 6 to 7 am/.append style={yellow!30}, segment 7 to 8 am/.append style={yellow!20}, segment 8 to 9 am/.append style={yellow!10},
        segment 9 to 10 pm/.append style={green!30}, segment 10 to 11 pm/.append style={green!20}, segment 11 pm to 12 am/.append style={green!10},
        label/.append style={text=green!40!black, font=\sffamily}, day/.append style={text=green!30!black}, weekday/.append style={text=olive!60!black}
    }"""
    },
    "Midnight Blue": {
        "chrono_color": "blue!80!black",
        "month_color": "blue!60!black",
        "style": r"""chronodex/mytheme/.style={
        24 hours, tick middle code={(0:\pgfkeysvalueof{/tikz/chronodex/tick middle start}) node[circle, inner sep=1.25pt, fill=cyan!80!black] {}},
        base ring/.append style={blue!70!black, very thick}, base segments/.append style={blue!2},
        primary ring/.append style={blue!80!black, thick}, primary tick/.append style={blue!80!black, thick}, secondary tick/.append style={blue!40!black, thick},
        segment 6 to 7 am/.append style={cyan!20}, segment 7 to 8 am/.append style={cyan!10}, segment 8 to 9 am/.append style={cyan!5},
        segment 9 to 10 pm/.append style={blue!30}, segment 10 to 11 pm/.append style={blue!20}, segment 11 pm to 12 am/.append style={blue!10},
        label/.append style={text=blue!80!black, font=\sffamily}, day/.append style={text=blue!90!black}, weekday/.append style={text=blue!60!black}
    }"""
    }
}

active_theme = THEMES.get(THEME_SELECTION, THEMES["Minimalist"])

# --- LaTeX Header Template ---
LATEX_HEADER_TEMPLATE = r"""
\documentclass[a4paper]{article}
\usepackage[margin=1cm]{geometry}
\usepackage{helvet}
\renewcommand{\familydefault}{\sfdefault}
\usepackage{tikz}
\usetikzlibrary{calendar, decorations.text}

\newcount\chronodexcurrentdate
\newcount\chronodexcurrentweekday

\tikzset{
    pics/chronodex/.style={
        code={
            \tikzset{chronodex/.cd, #1}
            \fill[chronodex/segment 6 to 7 am] (270:3) arc[start angle=270, end angle=240, radius=3] -- (240:2) arc[start angle=240, end angle=270, radius=2] -- cycle;
            \fill[chronodex/segment 7 to 8 am] (240:3) arc[start angle=240, end angle=210, radius=3] -- (210:2) arc[start angle=210, end angle=240, radius=2] -- cycle;
            \fill[chronodex/segment 8 to 9 am] (210:3) arc[start angle=210, end angle=180, radius=3] -- (180:2) arc[start angle=180, end angle=210, radius=2] -- cycle;
            \draw[chronodex/inner ring] (90:2) arc[start angle=90, end angle=-180, radius=2];
            \foreach \i [count=\a from 0] in {90,60,...,-180} {
                \draw[chronodex/inner line] (\i:3) -- (\i:2);
                \pgfmathparse{int(\a > 0 && \a < 7 ? 1 : 0)}
                \ifnum\pgfmathresult=1\relax \node[rotate={\i}, anchor=south east, chronodex/inner label] at (\i:3) {\a\pgfkeysvalueof{/tikz/chronodex/am suffix}}; \fi
                \pgfmathparse{int(\a > 6 && \a < 9 ? 1 : 0)}
                \ifnum\pgfmathresult=1\relax \node[rotate={\i-180}, anchor=south west, chronodex/inner label] at (\i:3) {\a\pgfkeysvalueof{/tikz/chronodex/am suffix}}; \fi
            }
            \fill[chronodex/segment 9 to 10 pm] (180:7) arc[start angle=180, end angle=150, radius=7] -- (150:6) arc[start angle=150, end angle=180, radius=6] -- cycle;
            \fill[chronodex/segment 10 to 11 pm] (150:7) arc[start angle=150, end angle=120, radius=7] -- (120:6) arc[start angle=120, end angle=150, radius=6] -- cycle;
            \fill[chronodex/segment 11 pm to 12 am] (120:7) arc[start angle=120, end angle=90, radius=7] -- (90:6) arc[start angle=90, end angle=120, radius=6] -- cycle;
            \draw[chronodex/outer ring] (180:7) arc[start angle=180, end angle=90, radius=7] (90:6) arc[start angle=90, end angle=180, radius=6];
            \foreach \o [count=\p from 0] in {180,150,...,90} {
                \draw[chronodex/outer line] (\o:7) -- (\o:6);
                \ifnum\p<3\relax
                    \pgfmathsetmacro{\h}{int(\p+9+\pgfkeysvalueof{/tikz/chronodex/24 hours conversion})}
                    \node[rotate={\o-180}, anchor=south west, chronodex/outer label] at (\o:7) {\h\pgfkeysvalueof{/tikz/chronodex/pm suffix}};
                \fi
            }
            \foreach \b [count=\a from 0] in {180,90,...,-90} {
                \draw[chronodex/additional ring] (\b:5) arc[start angle={\b}, end angle={\b-30}, radius=5] (\b:6) arc[start angle={\b}, end angle={\b-60}, radius=6] ({\b-30}:5) -- ({\b-30}:6);
                \fill[chronodex/base segments] (\b:4) arc[start angle={\b}, end angle={\b-30}, radius=4] -- ({\b-30}:5) arc[start angle={\b-30}, end angle={\b-60}, radius=5] -- ({\b-60}:6) arc[start angle={\b-60}, end angle={\b-90}, radius=6] -- ({\b-90}:3) arc[start angle={\b-90}, end angle={\b}, radius=3] -- (\b:3) -- cycle;
                \draw[chronodex/primary ring] (\b:3) -- (\b:4) arc[start angle={\b}, end angle={\b-30}, radius=4] -- ({\b-30}:3);
                \draw[chronodex/primary ring] ({\b-30}:3) -- ({\b-30}:5) arc[start angle={\b-30}, end angle={\b-60}, radius=5] -- ({\b-60}:3);
                \draw[chronodex/secondary ring] ({\b-30}:4) arc[start angle={\b-30}, end angle={\b-60}, radius=4];
                \draw[chronodex/primary ring] ({\b-60}:3) -- ({\b-60}:6) arc[start angle={\b-60}, end angle={\b-90}, radius=6] -- ({\b-90}:3);
                \draw[chronodex/secondary ring] ({\b-60}:4) arc[start angle={\b-60}, end angle={\b-90}, radius=4] ({\b-60}:5) arc[start angle={\b-60}, end angle={\b-90}, radius=5];
                \foreach \n in {7.5,15,22.5} {
                    \foreach \t/\a in {4/0, 5/30, 6/60} {
                        \tikzset{chronodex/tick bottom start={3}}
                        \draw[rotate={\b-\a-\n}, chronodex/primary tick] \pgfkeysvalueof{/tikz/chronodex/tick bottom code};
                        \tikzset{chronodex/tick top start={\t}}
                        \draw[rotate={\b-\a-\n}, chronodex/primary tick] \pgfkeysvalueof{/tikz/chronodex/tick top code};
                    }
                    \tikzset{chronodex/tick middle start={4}}
                    \draw[rotate={\b-30-\n}, chronodex/secondary tick] \pgfkeysvalueof{/tikz/chronodex/tick middle code};
                    \draw[rotate={\b-60-\n}, chronodex/secondary tick] \pgfkeysvalueof{/tikz/chronodex/tick middle code};
                    \tikzset{chronodex/tick middle start={5}}
                    \draw[rotate={\b-60-\n}, chronodex/secondary tick] \pgfkeysvalueof{/tikz/chronodex/tick middle code};
                }
                \ifnum\a=0\relax
                    \node[rotate={\b-180}, anchor=south west, chronodex/base label] at (\b:4) {9\pgfkeysvalueof{/tikz/chronodex/am suffix}};
                    \node[rotate={\b-210}, anchor=south west, chronodex/base label] at ({\b-30}:5) {10\pgfkeysvalueof{/tikz/chronodex/am suffix}};
                    \node[rotate={\b-240}, anchor=south west, chronodex/base label] at ({\b-60}:6) {11\pgfkeysvalueof{/tikz/chronodex/am suffix}};
                    \node[rotate={\b-270}, anchor=south west, chronodex/base label] at ({\b-90}:6) {12\pgfkeysvalueof{/tikz/chronodex/noon suffix}};
                \else
                    \ifnum\a=3\relax
                        \pgfmathsetmacro{\h}{int(\a*3-2+\pgfkeysvalueof{/tikz/chronodex/24 hours conversion})}
                        \node[rotate={\b-210}, anchor=south west, chronodex/base label] at ({\b-30}:5) {\h\pgfkeysvalueof{/tikz/chronodex/pm suffix}};
                        \pgfmathsetmacro{\h}{int(\a*3-1+\pgfkeysvalueof{/tikz/chronodex/24 hours conversion})}
                        \node[rotate={\b-240}, anchor=south west, chronodex/base label] at ({\b-60}:6) {\h\pgfkeysvalueof{/tikz/chronodex/pm suffix}};
                    \else
                        \pgfmathsetmacro{\h}{int(\a*3-2+\pgfkeysvalueof{/tikz/chronodex/24 hours conversion})}
                        \node[rotate={\b-30}, anchor=south east, chronodex/base label] at ({\b-30}:4) {\h\pgfkeysvalueof{/tikz/chronodex/pm suffix}};
                        \pgfmathsetmacro{\h}{int(\a*3-1+\pgfkeysvalueof{/tikz/chronodex/24 hours conversion})}
                        \node[rotate={\b-60}, anchor=south east, chronodex/base label] at ({\b-60}:5) {\h\pgfkeysvalueof{/tikz/chronodex/pm suffix}};
                        \pgfmathsetmacro{\h}{int(\a*3+\pgfkeysvalueof{/tikz/chronodex/24 hours conversion})}
                        \node[rotate={\b-90}, anchor=south east, chronodex/base label] at ({\b-90}:6) {\h\pgfkeysvalueof{/tikz/chronodex/pm suffix}};
                    \fi
                \fi
            }
            \draw[chronodex/base ring] (0:0) circle[radius=3];

            % Dynamically injected color for inner text
            \path [decorate, decoration={text along path, text={|\sffamily\bfseries\scriptsize\color{INJECT_CHRONO_COLOR}|CHRONODEX}, text align=center}] (180:2.5) arc [start angle=180, end angle=90, radius=2.5];

            \ifdefined\chronodexcurrentday
                \pgfmathtruncatemacro{\d}{\chronodexcurrentday.0}
                \node[chronodex/day] at (0,0) {\d};
                \node[chronodex/weekday] at (0,0) {\pgfcalendarweekdayname{\chronodexcurrentweekday}};
            \fi
        }
    },
    chronodex/date/.code={
        \pgfcalendardatetojulian{#1}{\chronodexcurrentdate}
        \pgfcalendarjuliantodate{\chronodexcurrentdate}{\chronodexcurrentyear}{\chronodexcurrentmonth}{\chronodexcurrentday}
        \pgfcalendarjuliantoweekday{\chronodexcurrentdate}{\chronodexcurrentweekday}
    },
    chronodex/date/.initial={},
    chronodex/am suffix/.initial={\,am},
    chronodex/pm suffix/.initial={\,pm},
    chronodex/noon suffix/.initial={\,noon},
    chronodex/24 hours conversion/.initial={0},
    chronodex/24 hours/.style={
        am suffix={:00}, pm suffix={:00}, noon suffix={:00}, 24 hours conversion={12},
    },
    chronodex/minor tick length/.initial={5},
    chronodex/major tick length/.initial={7},
    chronodex/tick top start/.initial={0},
    chronodex/tick middle start/.initial={0},
    chronodex/tick bottom start/.initial={0},
    chronodex/tick top code/.initial={
        (0:\pgfkeysvalueof{/tikz/chronodex/tick top start}) -- ++(0:{(\n == 15 ? \pgfkeysvalueof{/tikz/chronodex/major tick length} : \pgfkeysvalueof{/tikz/chronodex/minor tick length})*-1pt})
    },
    chronodex/tick middle code/.initial={
        ([shift={(0:{(\n == 15 ? \pgfkeysvalueof{/tikz/chronodex/major tick length} : \pgfkeysvalueof{/tikz/chronodex/minor tick length})*-0.5pt})}]0:\pgfkeysvalueof{/tikz/chronodex/tick middle start}) -- ++(0:{(\n == 15 ? \pgfkeysvalueof{/tikz/chronodex/major tick length} : \pgfkeysvalueof{/tikz/chronodex/minor tick length})*1pt})
    },
    chronodex/tick bottom code/.initial={
        (0:\pgfkeysvalueof{/tikz/chronodex/tick bottom start}) -- ++(0:{(\n == 15 ? \pgfkeysvalueof{/tikz/chronodex/major tick length} : \pgfkeysvalueof{/tikz/chronodex/minor tick length})*1pt})
    },
    chronodex/base ring/.style={thick},
    chronodex/base segments/.style={white!0},
    chronodex/primary ring/.style={thick},
    chronodex/primary tick/.style={},
    chronodex/secondary ring/.style={gray},
    chronodex/secondary tick/.style={},
    chronodex/inner ring/.style={gray, densely dashed},
    chronodex/inner line/.style={gray},
    chronodex/outer ring/.style={gray, densely dashed},
    chronodex/additional ring/.style={gray, densely dotted},
    chronodex/outer line/.style={gray},
    chronodex/label/.style={font=\footnotesize},
    chronodex/base label/.style={chronodex/label},
    chronodex/inner label/.style={chronodex/label, gray},
    chronodex/outer label/.style={chronodex/label, gray},
    chronodex/segment 6 to 7 am/.style={gray!25},
    chronodex/segment 7 to 8 am/.style={gray!15},
    chronodex/segment 8 to 9 am/.style={gray!5},
    chronodex/segment 9 to 10 pm/.style={white!0},
    chronodex/segment 10 to 11 pm/.style={white!0},
    chronodex/segment 11 pm to 12 am/.style={white!0},
    chronodex/day/.style={anchor=south, font=\fontsize{45}{50}\selectfont\bfseries},
    chronodex/weekday/.style={anchor=north, font=\fontsize{20}{22}\selectfont},
    INJECT_THEME_STYLE
}
"""

# --- Execution ---
print(f"🚀 Generating Planner: {FILE_NAME}.pdf")
print(f"📅 Dates: {START_DATE} to {END_DATE}")
print(f"🎨 Theme: {THEME_SELECTION}")

latex_header = LATEX_HEADER_TEMPLATE.replace("INJECT_THEME_STYLE", active_theme["style"])
latex_header = latex_header.replace("INJECT_CHRONO_COLOR", active_theme["chrono_color"])

start_date = datetime.datetime.strptime(START_DATE, "%Y-%m-%d").date()
end_date = datetime.datetime.strptime(END_DATE, "%Y-%m-%d").date()
dates = [start_date + datetime.timedelta(days=i) for i in range((end_date - start_date).days + 1)]

tikz_blocks = ""
for i, current_date in enumerate(dates):
    date_str = current_date.strftime("%Y-%m-%d")
    month_year_str = current_date.strftime("%b '%y")
    if i % 6 == 0:
        if i != 0: tikz_blocks += "\\clearpage\n"
        tikz_blocks += "\\vspace*{\\fill}\n\\noindent\n"
    tikz_blocks += f"\\begin{{minipage}}[c]{{0.48\\textwidth}}\n  \\centering\n  \\begin{{tikzpicture}}[scale=0.45, transform shape]\n    \\pic {{chronodex={{date={date_str}, mytheme}}}};\n    \\draw[dashed, thin, gray] (0:0) circle[radius=8.8];\n    \\node[text={active_theme['month_color']}, font=\\sffamily\\fontsize{{14}}{{16}}\\selectfont, anchor=north] at (0,-0.9) {{{month_year_str}}};\n  \\end{{tikzpicture}}\n\\end{{minipage}}%"
    if i % 2 == 0:
        if i != len(dates) - 1: tikz_blocks += "\\hfill\n"
    else:
        if i % 6 != 5 and i != len(dates) - 1: tikz_blocks += "\n\\par\\vspace{\\fill}\n\\noindent\n"
    if i % 6 == 5 or i == len(dates) - 1: tikz_blocks += "\n\\par\\vspace*{\\fill}\n"

latex_document = latex_header + r"\begin{document}" + tikz_blocks + r"\end{document}"
with open(f"{FILE_NAME}.tex", "w") as f: f.write(latex_document.replace('\xa0', ' '))

process = subprocess.run(["pdflatex", "-interaction=nonstopmode", f"{FILE_NAME}.tex"], capture_output=True, text=True)
if process.returncode == 0:
    subprocess.run(["pdflatex", "-interaction=nonstopmode", f"{FILE_NAME}.tex"], capture_output=True)
    print(f"✅ Success! Generated {FILE_NAME}.pdf")
    try:
        from google.colab import files
        files.download(f"{FILE_NAME}.pdf")
    except: pass
else:
    print("❌ Compilation Failed. Check your date format (YYYY-MM-DD).")

🚀 Generating Planner: Chronodex_Planner_Final.pdf
📅 Dates: 2026-06-05 to 2026-06-30
🎨 Theme: Corporate Slate
✅ Success! Generated Chronodex_Planner_Final.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>